In [3]:
# ============================================================
# DATA 622 - Homework 5: Text Summarization
# ============================================================

# ── CELL 1: Install & Setup ───────────────────────────────────
!pip install -q --upgrade transformers nltk torch

import nltk, math, numpy as np, warnings
from collections import Counter
warnings.filterwarnings("ignore")

# Fix for newer NLTK versions - download all needed resources
for resource in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(resource, quiet=True)

# If punkt_tab still fails, use this manual sentence splitter fallback
import re
from nltk.corpus import stopwords

def safe_sent_tokenize(text):
    try:
        from nltk.tokenize import sent_tokenize
        return sent_tokenize(text)
    except LookupError:
        # Regex-based fallback sentence splitter
        return re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())

def word_tokenize_simple(text):
    try:
        from nltk.tokenize import word_tokenize
        return word_tokenize(text)
    except LookupError:
        return re.findall(r'\b[a-zA-Z]+\b', text)

# ─────────────────────────────────────────────
# ARTICLE TEXT
# ─────────────────────────────────────────────
article_text = """Defense Secretary Pete Hegseth indicated that the Pentagon has established strategies for various potential scenarios in Greenland, which may include an invasion of the territory. During a House Armed Services Committee session on June 12, Republican Representative Mike Turner pressed Hegseth to clarify whether there are indeed plans to invade Greenland. Hegseth responded, stating "The Pentagon has plans for any number of contingencies." Turner persisted, asking, "It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland? Because I sure as hell hope that is not your testimony." In response, Hegseth assured, "We look forward to working with Greenland to ensure that it is secured from any potential threats." President Donald Trump has not dismissed the possibility of using force in his ambition to acquire Greenland, although he has mentioned that such measures may not be necessary. He has argued that obtaining Greenland is crucial for national security, pointing to the increasing influence of China and Russia in the area. Additionally, the island is abundant in essential minerals, which the U.S. aims to secure in order to counter Chinese dominance in certain sectors. In a March visit to Pituffik Space Base, a U.S. installation in Greenland, Vice President JD Vance criticized Denmark for allegedly failing to safeguard the Arctic territory while minimizing Trump's threats of military action. Danish officials have responded firmly, with Prime Minister Mette Frederiksen stating, "The U.S. shall not take over. Greenland belongs to the Greenlanders," following Vance's visit. In a recent development that further distances the U.S. from Denmark and its European allies, reports suggest that the Pentagon intends to transfer its management of Greenland from U.S. European Command to U.S. Northern Command. Rep. Adam Smith of Washington, the top Democrat on the committee, asked Hegseth whether it was the policy of the Pentagon that the U.S. needs to be prepared to take Greenland and Panama by force. Hegseth said, "Our job at the DOD is to have plans. I think the American people would want the Pentagon to have plans for any particular contingency and thankfully we are in the planning business." Democrats on the panel scoffed at those answers. Rep. Salud Carbajal, a Democrat from California, exclaimed, "You are unfit to lead. You should just get the hell out." Hegseth's use of Signal chats to discuss U.S. military actions against Houthi rebels in Yemen also came under fire, with lawmakers questioning whether he disclosed classified information."""

sentences = safe_sent_tokenize(article_text)
stop_words = set(stopwords.words('english'))
print(f"Total sentences in article: {len(sentences)}\n")

# ─────────────────────────────────────────────
# Q1: TextRank Extractive Summary
# ─────────────────────────────────────────────
print("=" * 60)
print("Q1: TextRank Extractive Summary")
print("=" * 60)

def sentence_similarity(s1, s2):
    words1 = set(w.lower() for w in word_tokenize_simple(s1) if w.isalpha() and w.lower() not in stop_words)
    words2 = set(w.lower() for w in word_tokenize_simple(s2) if w.isalpha() and w.lower() not in stop_words)
    if not words1 or not words2:
        return 0.0
    return len(words1 & words2) / (math.log(len(words1) + 1) + math.log(len(words2) + 1))

n = len(sentences)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            sim_matrix[i][j] = sentence_similarity(sentences[i], sentences[j])

row_sums = sim_matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
sim_matrix /= row_sums

damping = 0.85
scores = np.ones(n) / n
for _ in range(100):
    scores = (1 - damping) / n + damping * sim_matrix.T.dot(scores)

top_indices = sorted(np.argsort(scores)[-3:])
print("Top 3 sentences by TextRank:\n")
for i in top_indices:
    print(f"  [{i+1}] {sentences[i].strip()}\n")

# ─────────────────────────────────────────────
# Q2: Frequency-Based Sentence Scoring
# ─────────────────────────────────────────────
print("=" * 60)
print("Q2: Frequency-Based Sentence Scoring Summary")
print("=" * 60)

words = [w.lower() for w in word_tokenize_simple(article_text) if w.isalpha() and w.lower() not in stop_words]
word_freq = Counter(words)
max_freq = max(word_freq.values())
word_freq = {w: f / max_freq for w, f in word_freq.items()}

sentence_scores = {}
for sent in sentences:
    sentence_scores[sent] = sum(word_freq.get(w.lower(), 0) for w in word_tokenize_simple(sent) if w.isalpha())

top3_freq = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:3]
top3_freq_ordered = sorted(top3_freq, key=lambda s: sentences.index(s))
print("Top 3 sentences by frequency scoring:\n")
for s in top3_freq_ordered:
    print(f"  - {s.strip()}\n")

# ─────────────────────────────────────────────
# Q3: Abstractive Summary using BART
# ─────────────────────────────────────────────
print("=" * 60)
print("Q3: Abstractive Summary using BART (facebook/bart-large-cnn)")
print("=" * 60)

from transformers import BartForConditionalGeneration, BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

inputs = tokenizer(article_text, return_tensors="pt", max_length=1024, truncation=True)
ids = model.generate(inputs["input_ids"], max_length=120, min_length=40,
                     length_penalty=2.0, num_beams=4, early_stopping=True)
print("\nBART Abstractive Summary:\n")
print(tokenizer.decode(ids[0], skip_special_tokens=True))

# ─────────────────────────────────────────────
# Q4: Lead-3 Summary
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("Q4: Lead-3 Summary (First 3 Sentences)")
print("=" * 60 + "\n")
for i, sent in enumerate(sentences[:3], 1):
    print(f"  [{i}] {sent.strip()}\n")

# ─────────────────────────────────────────────
# Q5: Manual Compression (~20%)
# ─────────────────────────────────────────────
print("=" * 60)
print("Q5: Manual Compression Summary (~20% of sentences)")
print("=" * 60)
k = max(1, round(len(sentences) * 0.20))
print(f"\nSelecting {k} out of {len(sentences)} sentences (~20%):\n")
top_k = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:k]
top_k_ordered = sorted(top_k, key=lambda s: sentences.index(s))
for s in top_k_ordered:
    print(f"  - {s.strip()}\n")

# ─────────────────────────────────────────────
# Q6: LLM Summary using Flan-T5 (no API key needed)
# ─────────────────────────────────────────────
print("=" * 60)
print("Q6: LLM Summary using google/flan-t5-base")
print("=" * 60)

from transformers import T5ForConditionalGeneration, T5Tokenizer

t5_tok = T5Tokenizer.from_pretrained("google/flan-t5-base")
t5_mod = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

prompt = f"Summarize the following news article in 3-4 sentences:\n\n{article_text}"
t5_in = t5_tok(prompt, return_tensors="pt", max_length=512, truncation=True)
t5_out = t5_mod.generate(t5_in["input_ids"], max_length=150, min_length=40,
                          length_penalty=2.0, num_beams=4, early_stopping=True)
print("\nFlan-T5 LLM Summary:\n")
print(t5_tok.decode(t5_out[0], skip_special_tokens=True))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.0 MB/s eta 0:00:00
Total sentences in article: 20

Q1: TextRank Extractive Summary
Top 3 sentences by TextRank:

  [1] Defense Secretary Pete Hegseth indicated that the Pentagon has established strategies for various potential scenarios in Greenland, which may include an invasion of the territory.

  [4] Turner persisted, asking, "It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland?

  [14] Rep. Adam Smith of Washington, the top Democrat on the committee, asked Hegseth whether it was the policy of the Pentagon that the U.S. needs to be prepared to take Greenland and Panama by force.

Q2: Frequency-Based Sentence Scoring Summary
Top 3 sentences by frequency scoring:

  - During a House Armed Services Committee session on June 12, Republican Representative Mike Turner pressed Hegseth to clarify whether there are indeed plans to invade Greenland.

  - In a March visit to Pit

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]


BART Abstractive Summary:

Defense Secretary Pete Hegseth indicated that the Pentagon has established strategies for various potential scenarios in Greenland. President Donald Trump has not dismissed the possibility of using force in his ambition to acquire Greenland. He has argued that obtaining Greenland is crucial for national security.

Q4: Lead-3 Summary (First 3 Sentences)

  [1] Defense Secretary Pete Hegseth indicated that the Pentagon has established strategies for various potential scenarios in Greenland, which may include an invasion of the territory.

  [2] During a House Armed Services Committee session on June 12, Republican Representative Mike Turner pressed Hegseth to clarify whether there are indeed plans to invade Greenland.

  [3] Hegseth responded, stating "The Pentagon has plans for any number of contingencies."

Q5: Manual Compression Summary (~20% of sentences)

Selecting 4 out of 20 sentences (~20%):

  - Defense Secretary Pete Hegseth indicated that the Pentag

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Flan-T5 LLM Summary:

The U.S. is prepared to take Greenland and Panama by force, and we are prepared to take Greenland and Panama by force," Defense Secretary Pete Hegseth said in a statement.
